# Evaluation with Additional Metrics

In this notebook, we will:

- Perform retrieval within the matches dataset.
- Compute Precision, Recall, F1-score, Average Accuracy, Case Accuracy, and Mean Average Precision (mAP).
- Save the evaluation results in a DataFrame
- Identify common failure cases for further analysis.
- Clear memory after processing to manage resources efficiently.
---

## **Step 1: Import Libraries**


We begin by importing the necessary libraries.

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
import gc

---

## **Step 2: Load Embeddings and Data**

We load the embeddings and related data saved from the previous notebook.

In [2]:
# Load embeddings and data
embeddings = np.load('matches_embeddings.npy')
image_files = np.load('matches_image_files.npy', allow_pickle=True)
base_ids = np.load('matches_base_ids.npy', allow_pickle=True)

---

## **Step 3: Define Evaluation Functions**

We define functions to calculate the evaluation metrics.

In [3]:
# Function to calculate evaluation metrics
def calculate_metrics(retrieved_base_ids, query_base_id, expected_base_ids, k):
    y_true = [1 if base_id == query_base_id else 0 for base_id in retrieved_base_ids[:k]]
    y_scores = np.arange(k, 0, -1)  # Higher scores for higher ranks

    # Precision, Recall, F1-score
    precision = precision_score([1]*len(y_true), y_true, zero_division=0)
    recall = recall_score([1]*len(y_true), y_true, zero_division=0)
    f1 = f1_score([1]*len(y_true), y_true, zero_division=0)
    
    # Average Precision (for mAP)
    average_precision = average_precision_score([1]*len(y_true), y_scores)

    # Average Accuracy
    average_accuracy = sum(y_true) / k

    # Case Accuracy
    retrieved_set = set(retrieved_base_ids[:k])
    expected_set = set(expected_base_ids)
    missing = expected_set - retrieved_set
    case_accuracy = 1.0 if not missing else 1 - len(missing) / len(expected_set)

    return precision, recall, f1, average_precision, average_accuracy, case_accuracy

---

## **Step 4: Perform Retrieval and Compute Metrics**

We perform retrieval for each image and compute the evaluation metrics.

In [4]:
# Initialize results list
results = []

# Set the number of top results to consider
k = 5  # You can adjust k as needed

# Iterate over each image
for i in tqdm(range(len(embeddings)), desc='Evaluating'):
    query_embedding = embeddings[i]
    query_base_id = base_ids[i]
    query_image_file = image_files[i]

    # Compute similarities with all embeddings
    similarities = np.dot(embeddings, query_embedding)
    similarities[i] = -np.inf  # Exclude the query image itself

    # Get top K indices
    top_k_indices = np.argsort(-similarities)[:k]
    retrieved_base_ids = [base_ids[idx] for idx in top_k_indices]

    # Expected matches (excluding the query image)
    expected_base_ids = [base_ids[j] for j in range(len(base_ids)) if base_ids[j] == query_base_id and j != i]
    num_matches = len(expected_base_ids)

    # Compute metrics
    precision, recall, f1, average_precision, avg_accuracy, case_accuracy = calculate_metrics(
        retrieved_base_ids, query_base_id, expected_base_ids, k)

    # Store results
    results.append({
        'Query Image': query_image_file,
        'Number of Matches': num_matches,
        'Precision': round(precision, 2),
        'Recall': round(recall, 2),
        'F1-score': round(f1, 2),
        'Average Precision': round(average_precision, 2),
        'Average Accuracy': round(avg_accuracy, 2),
        'Case Accuracy': round(case_accuracy, 2)
    })

Evaluating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 235.98it/s]


---

## **Step 5: Create Evaluation Table**

We create a DataFrame to display the evaluation results.

In [5]:
# Create DataFrame from results
df_results = pd.DataFrame(results)
df_results

,Query Image,Number of Matches,Precision,Recall,F1-score,Average Precision,Average Accuracy,Case Accuracy
0,1353_04.png,4,1.0,0.4,0.57,1.0,0.4,1.0
1,2012_02.jpg,3,1.0,0.2,0.33,1.0,0.2,1.0
2,1353_05.jpg,4,0.0,0.0,0.00,1.0,0.0,0.0
3,2012_02.png,3,1.0,0.2,0.33,1.0,0.2,1.0
4,387_04.jpg,4,1.0,0.6,0.75,1.0,0.6,1.0
5,387_05.jpg,4,0.0,0.0,0.00,1.0,0.0,0.0
6,2012_01.jpg,3,1.0,0.2,0.33,1.0,0.2,1.0
7,2012_01.png,3,1.0,0.2,0.33,1.0,0.2,1.0
8,1353_02.jpg,4,1.0,0.2,0.33,1.0,0.2,1.0
9,387_01.jpg,4,1.0,0.4,0.57,1.0,0.4,1.0


---

## **Step 6: Save Evaluation Results**

We save the evaluation results to a CSV file.

In [6]:
# Save evaluation results to CSV
df_results.to_csv('evaluation_results.csv', index=False)
print('Evaluation results saved.')

Evaluation results saved.


---

## **Step 7: Identify Common Failure Cases**

We analyze images with lower accuracy to identify common failure cases.

In [7]:
# Filter results with low F1-score
low_performance = df_results[df_results['F1-score'] < 0.5]
print('Images with low F1-score:')
print(low_performance[['Query Image', 'F1-score']])

Images with low F1-score:
    Query Image  F1-score
1   2012_02.jpg      0.33
2   1353_05.jpg      0.00
3   2012_02.png      0.33
5    387_05.jpg      0.00
6   2012_01.jpg      0.33
7   2012_01.png      0.33
8   1353_02.jpg      0.33
10  1353_03.png      0.33
14   722_01.jpg      0.33
17   464_01.jpg      0.00
18  2823_02.jpg      0.33
19  2823_03.jpg      0.33
21   722_04.jpg      0.00
22  2823_01.jpg      0.33
23   464_03.jpg      0.33
25  2871_05.jpg      0.00
26  3253_05.jpg      0.33
30  2871_04.jpg      0.33
32  2257_01.jpg      0.33
35  3253_03.PNG      0.33
39  2871_02.jpg      0.33
40  3253_02.jpg      0.33
42  3216_04.jpg      0.33
44  3253_01.jpg      0.33
46  1024_03.jpg      0.33
47  1024_02.jpg      0.33
50  1024_04.jpg      0.00


---

## **Step 8: Clear Memory**

We clear variables and free up memory.

In [8]:
# Clear variables and free memory
del embeddings, image_files, base_ids, results
gc.collect()

0

---

## **Step 9: Conclusion**

We have successfully evaluated the retrieval performance with additional metrics and identified common failure cases.